### Imports

In [31]:
import os
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

#### Configuration

In [36]:
STYLE = "style1"
CONFIG = {
    "content_layer_logs": {
        "conv2_2": f"../results/content_layer_depth/logs/{STYLE}_2_2.csv",
        "conv3_3": f"../results/content_layer_depth/logs/{STYLE}_3_3.csv",
        "conv4_2": f"../results/content_layer_depth/logs/{STYLE}_4_2.csv",
        "conv5_2": f"../results/content_layer_depth/logs/{STYLE}_5_2.csv",
    },

    "alpha_beta_logs": {
        "α=1, β=1e3":  f"../results/alpha_beta_ratio/logs/{STYLE}_10e3.csv",
        "α=1, β=1e4":  f"../results/alpha_beta_ratio/logs/{STYLE}_10e4.csv",
        "α=1, β=1e5":  f"../results/alpha_beta_ratio/logs/{STYLE}_10e5.csv",
        "α=1, β=1e6":  f"../results/alpha_beta_ratio/logs/{STYLE}_10e6.csv",
        "α=1, β=1e7":  f"../results/alpha_beta_ratio/logs/{STYLE}_10e7.csv",
    },

    "style_layer_logs": {
        "conv1_1": f"../results/style_layer_depth/logs/{STYLE}_1_1.csv",
        "conv2_1": f"../results/style_layer_depth/logs/{STYLE}_2_1.csv",
        "conv3_1": f"../results/style_layer_depth/logs/{STYLE}_3_1.csv",
        "conv4_1": f"../results/style_layer_depth/logs/{STYLE}_4_1.csv",
        "conv5_1": f"../results/style_layer_depth/logs/{STYLE}_5_1.csv",
    },
    "output_dir": f"../results/plots/{STYLE}",
    "figure_dpi": 150,
    "style": "seaborn-v0_8-paper",
}
PALETTE = [ "#ff6b35", "#4cc9f0", "#f72585", "#7fff7f", "#ffbe0b", "#8338ec", "#fb5607", "#3a86ff"]

plt.style.use(CONFIG["style"])
plt.rcParams.update({
    "font.family":       "monospace",
    "axes.titlesize":    11,
    "axes.labelsize":    9,
    "xtick.labelsize":   8,
    "ytick.labelsize":   8,
    "axes.grid":         True,
    "grid.alpha":        0.15,
    "legend.fontsize":   8,
    "figure.facecolor":  "#0d0d10" if CONFIG["style"] == "dark_background" else "white",
    "axes.facecolor":    "#16161c" if CONFIG["style"] == "dark_background" else "#f9f9f9",
})

#### Helpers

In [17]:
def load_logs(log_dict):
    loaded = {}
    for label, path in log_dict.items():
        if os.path.exists(path):
            df = pd.read_csv(path)
            loaded[label] = df
    return loaded

def save(fig, name: str):
    out = Path(CONFIG["output_dir"])
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    fig.savefig(path, dpi=CONFIG["figure_dpi"], bbox_inches="tight")
    print(f"  Saved: {path}")
    plt.close(fig)

#### Graphs

In [18]:
def plot_phase_zoom(logs: dict[str, pd.DataFrame], early_end: int = 300, key_layer: str = None, save_graph: bool = False):
    label = key_layer if key_layer in logs else list(logs.keys())[0]
    df = logs[label]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    early = df[df["step"] <= early_end]
    late  = df[df["step"] >  early_end]

    for ax, data, phase in [(ax1, early, f"Early Phase (0–{early_end})"),
                             (ax2, late,  f"Late Phase ({early_end}+)")]:
        ax.semilogy(data["step"], data["total_loss"],   color=PALETTE[0], lw=1.4, label="Total")
        ax.semilogy(data["step"], data["content_loss"], color=PALETTE[1], lw=1.2, label="Content", linestyle="--")
        ax.semilogy(data["step"], data["style_loss"],   color=PALETTE[2], lw=1.2, label="Style",   linestyle=":")
        ax.set_title(f"{phase}  [{label}]")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss (log)")
        ax.legend()

    fig.suptitle("Optimization Phases", fontsize=13, fontweight="bold")
    fig.tight_layout()
    if save_graph:
        save(fig, "phase_zoom")


In [19]:
def plot_single_run_dashboard(df: pd.DataFrame, run_label: str = "conv4_2", save_graph: bool = False):
    fig = plt.figure(figsize=(14, 9))
    gs = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

    ax_total   = fig.add_subplot(gs[0, 0])
    ax_cs      = fig.add_subplot(gs[0, 1])
    ax_ratio   = fig.add_subplot(gs[1, 0])
    ax_delta   = fig.add_subplot(gs[1, 1])

    steps = df["step"]

    ax_total.semilogy(steps, df["total_loss"], color=PALETTE[0], lw=1.5)
    ax_total.set_title("Total Loss (log)")
    ax_total.set_xlabel("Step"); ax_total.set_ylabel("Loss")

    ax_cs.semilogy(steps, df["content_loss"], color=PALETTE[1], lw=1.4, label="Content")
    ax_cs.semilogy(steps, df["style_loss"],   color=PALETTE[2], lw=1.4, label="Style", linestyle="--")
    ax_cs.set_title("Content vs Style Loss (log)")
    ax_cs.set_xlabel("Step"); ax_cs.set_ylabel("Loss")
    ax_cs.legend()

    ratio = df["content_loss"] / df["style_loss"].replace(0, np.nan)
    ax_ratio.semilogy(steps, ratio, color=PALETTE[3], lw=1.4)
    ax_ratio.set_title("Content / Style Ratio (log)")
    ax_ratio.set_xlabel("Step"); ax_ratio.set_ylabel("Ratio")

    delta = df["total_loss"].diff().abs()
    ax_delta.semilogy(steps[1:], delta[1:], color=PALETTE[4], lw=1.0, alpha=0.8)
    ax_delta.set_title("|ΔLoss| per Step — Gradient Activity")
    ax_delta.set_xlabel("Step"); ax_delta.set_ylabel("|ΔLoss| (log)")

    fig.suptitle(f"Detailed Run Dashboard — {run_label}", fontsize=13, fontweight="bold")
    fname = f"dashboard_{run_label.replace(' ', '_').replace('/', '_')}"
    if save_graph:
        save(fig, fname)


In [20]:
def plot_convergence_heatmap(logs: dict[str, pd.DataFrame], checkpoints=(50, 100, 300, 1000, 3000, 6000), save_graph: bool = False):
    labels = list(logs.keys())
    data = []
    available_ckpts = []

    for ckpt in checkpoints:
        col = []
        for df in logs.values():
            row = df[df["step"] <= ckpt]
            col.append(row["total_loss"].iloc[-1] if not row.empty else np.nan)
        if not all(np.isnan(col)):
            data.append(col)
            available_ckpts.append(ckpt)

    matrix = np.array(data).T 

    fig, ax = plt.subplots(figsize=(max(8, len(available_ckpts) * 1.4), max(3, len(labels) * 0.8 + 1.5)))
    im = ax.imshow(matrix, aspect="auto", cmap="inferno",
                   norm=plt.matplotlib.colors.LogNorm(
                       vmin=np.nanmin(matrix[matrix > 0]),
                       vmax=np.nanmax(matrix)))

    ax.set_xticks(range(len(available_ckpts)))
    ax.set_xticklabels([str(c) for c in available_ckpts])
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Step Checkpoint")
    ax.set_ylabel("Content Layer")
    ax.set_title("Total Loss Heatmap (log scale) — Convergence Overview")

    for r in range(len(labels)):
        for c in range(len(available_ckpts)):
            val = matrix[r, c]
            if not np.isnan(val):
                ax.text(c, r, f"{val:.1f}", ha="center", va="center",
                        fontsize=7, color="white" if val < np.nanmedian(matrix) else "black")

    fig.colorbar(im, ax=ax, label="Total Loss (log)")
    fig.tight_layout()
    if save_graph:
        save(fig, "convergence_heatmap")


In [21]:
def plot_style_layer_comparison(logs: dict[str, pd.DataFrame], save_graph: bool = False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for i, (label, df) in enumerate(logs.items()):
        c = PALETTE[i % len(PALETTE)]
        axes[0].semilogy(df["step"], df["style_loss"],   color=c, lw=1.4, label=label)
        axes[1].semilogy(df["step"], df["content_loss"], color=c, lw=1.4, label=label)

    axes[0].set_title("Style Loss by Style Layer Config")
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Style Loss (log)")
    axes[0].legend()

    axes[1].set_title("Content Loss by Style Layer Config")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Content Loss (log)")
    axes[1].legend()

    fig.suptitle("Style Layer Depth Experiment", fontsize=13, fontweight="bold")
    fig.tight_layout()
    if save_graph:
        save(fig, "style_layer_comparison")


In [22]:
def plot_alpha_beta_sweep(logs: dict[str, pd.DataFrame], save_graph: bool = False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for i, (label, df) in enumerate(logs.items()):
        c = PALETTE[i % len(PALETTE)]
        axes[0].semilogy(df["step"], df["total_loss"],
                         color=c, lw=1.4, label=label, alpha=0.9)
        axes[1].semilogy(df["step"], df["content_loss"] / df["total_loss"].replace(0, np.nan),
                         color=c, lw=1.4, label=label, alpha=0.9)

    axes[0].set_title("Total Loss — α/β Sweep")
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss (log)")
    axes[0].legend()

    axes[1].set_title("Content Loss Fraction of Total")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("content / total")
    axes[1].legend()

    fig.suptitle("Effect of α / β Weighting", fontsize=13, fontweight="bold")
    fig.tight_layout()
    if save_graph:
        save(fig, "alpha_beta_sweep")


In [25]:
def plot_ab_final_scatter(logs: dict[str, pd.DataFrame], save_graph: bool = False):
    fig, ax = plt.subplots(figsize=(8, 6))

    for i, (label, df) in enumerate(logs.items()):
        fc = df["content_loss"].iloc[-1]
        fs = df["style_loss"].iloc[-1]
        ax.scatter(fs, fc, color=PALETTE[i % len(PALETTE)], s=120, zorder=5)
        ax.annotate(label, (fs, fc),
                    textcoords="offset points", xytext=(6, 4),
                    fontsize=8, color=PALETTE[i % len(PALETTE)])

    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Final Style Loss (log)")
    ax.set_ylabel("Final Content Loss (log)")
    ax.set_title("Content–Style Trade-off Frontier\n(final step, per α/β config)")
    fig.tight_layout()
    if save_graph:
        save(fig, "ab_final_scatter")


_Note_: The basic structure of visualizations were consulted with the help of the digital assistant Claude (Anthropic).

_Role of the assistant_: basic structure of matplotlib code.

All modifications, parameter tuning and integration withown data were done independently.

In [37]:
content_logs = load_logs(CONFIG["content_layer_logs"])
ab_logs = load_logs(CONFIG["alpha_beta_logs"])
style_logs = load_logs(CONFIG["style_layer_logs"])

plot_phase_zoom(content_logs, save_graph = True)
for label, df in content_logs.items():
    plot_single_run_dashboard(df, run_label=label, save_graph = True)

plot_convergence_heatmap(content_logs, save_graph = True)
plot_style_layer_comparison(style_logs, save_graph = True)
plot_alpha_beta_sweep(ab_logs, save_graph = True)
plot_ab_final_scatter(ab_logs, save_graph = True)

  Saved: ..\results\plots\style1\phase_zoom.png
  Saved: ..\results\plots\style1\dashboard_conv2_2.png
  Saved: ..\results\plots\style1\dashboard_conv3_3.png
  Saved: ..\results\plots\style1\dashboard_conv4_2.png
  Saved: ..\results\plots\style1\dashboard_conv5_2.png
  Saved: ..\results\plots\style1\convergence_heatmap.png
  Saved: ..\results\plots\style1\style_layer_comparison.png
  Saved: ..\results\plots\style1\alpha_beta_sweep.png
  Saved: ..\results\plots\style1\ab_final_scatter.png
